# 6. Plotly Visual Analytics and Dash App

In [2]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

# Check if the file is in the data folder or in the root directory
cleaned_path = DATA_DIR / "spacex_launches_cleaned.csv"
if not cleaned_path.exists():
    cleaned_path = PROJECT_DIR / "spacex_launches_cleaned.csv"

if not cleaned_path.exists():
    raise FileNotFoundError(f"spacex_launches_cleaned.csv not found in {DATA_DIR} or {PROJECT_DIR}. Run notebooks 01 and 02 successfully first.")

df = pd.read_csv(cleaned_path)
df["outcome"] = df["success"].map({1:"Success", 0:"Failure"})

In [3]:
# Plot 1: known launch outcomes
fig1 = px.pie(df.dropna(subset=["outcome"]), names="outcome",
              title="Distribution of Known Launch Outcomes")
fig1.show()

In [8]:
# Plot 2: Launch outcomes grouped by rocket type
if "rocket" in df.columns:
    plot_df = df.dropna(subset=["rocket", "outcome"])
    fig2 = px.histogram(plot_df, x="rocket", color="outcome", barmode="group",
                        title="Launch Outcomes by Rocket Type",
                        labels={"rocket": "Rocket Type", "count": "Number of Launches"})
    fig2.show()
else:
    print("Error: 'rocket' column not found in DataFrame. Available columns are:", list(df.columns))

In [6]:
# Plot 3: launch counts by year
year_df = df.dropna(subset=["year"]).groupby("year").size().reset_index(name="launch_count")
fig3 = px.bar(year_df, x="year", y="launch_count", title="Launches by Year")
fig3.show()

In [7]:
# Generate a Dash app file to run locally. Install dash: pip install dash
app_path = PROJECT_DIR / "src" / "app.py"
app_path.parent.mkdir(exist_ok=True)
app_source = r'''
from pathlib import Path
import pandas as pd
from dash import Dash, dcc, html
import plotly.express as px

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data" / "spacex_launches_cleaned.csv"
df = pd.read_csv(DATA)
df["outcome"] = df["success"].map({1:"Success", 0:"Failure"})
fig1 = px.pie(df.dropna(subset=["outcome"]), names="outcome", title="Known Launch Outcomes")
fig2 = px.scatter(df.dropna(subset=["payload_mass_kg","outcome"]),
                  x="payload_mass_kg", y="flight_number", color="outcome",
                  hover_name="name", title="Payload Mass vs Launch Outcome")
year_df = df.dropna(subset=["year"]).groupby("year").size().reset_index(name="launch_count")
fig3 = px.bar(year_df, x="year", y="launch_count", title="Launches by Year")
app = Dash(__name__)
app.layout = html.Div([
    html.H1("SpaceX Launch Analytics Dashboard"),
    html.P("Exploratory analysis of historical SpaceX launch records."),
    dcc.Graph(figure=fig1),
    dcc.Graph(figure=fig2),
    dcc.Graph(figure=fig3),
], style={"maxWidth":"1200px","margin":"auto","fontFamily":"Arial"})
if __name__ == "__main__":
    app.run(debug=True)
'''
app_path.write_text(app_source, encoding="utf-8")
print("Created:", app_path)

Created: /content/src/app.py


## Run Dash locally
From the project root, install dependencies with `pip install -r requirements.txt`, then run `python src/app.py`. Open the local address shown in the terminal. Capture screenshots of the running app for the presentation.